In [35]:
"""
Estrae da OUT_VARIATIONS_STANDARD_RUN2_DATA.root i primitives contenuti
in ciascun TCanvas (c_fit_1 ... c_fit_24): 2 TGraph, 1 TF1, 1 TH1.
Richiede PyROOT installato (root_numpy non necessario).

Uso:
    python3 extract_canvas_plots.py
"""

import ROOT
import numpy as np
import matplotlib.pyplot as plt
import os
import re

#INPUT_FILE = "ROOT_TREES_DEDX/OUT_VARIATIONS_STANDARD_RUN2_DATA.root"
#FIT_PARAMS_FILE = "fitDATA.txt"  # risultati del fit (prima convoluzione landau-gauss)
#OUT_DIR = "extracted_plots_DATA"

INPUT_FILE = "ROOT_TREES_DEDX/OUT_VARIATIONS_STANDARD_RUN2.root"
FIT_PARAMS_FILE = "fitMC.txt"  # risultati del fit (prima convoluzione landau-gauss)
OUT_DIR = "extracted_plots_MC"

os.makedirs(OUT_DIR, exist_ok=True)

f = ROOT.TFile.Open(INPUT_FILE)
if not f or f.IsZombie():
    raise RuntimeError(f"Impossibile aprire {INPUT_FILE}")

canvas_names = [k.GetName() for k in f.GetListOfKeys() if k.GetClassName() == "TCanvas"]
print(f"Trovati {len(canvas_names)} canvas: {canvas_names}")


def load_fit_params(path):
    """Legge fitDATA.txt (header su riga 1, colonne separate da spazi) e
    ritorna un dict {rr: {nome_colonna: valore}}."""
    if not os.path.exists(path):
        print(f"ATTENZIONE: {path} non trovato, la legenda non includera' i parametri del fit")
        return {}
    with open(path) as fh:
        header = fh.readline().split()
        params = {}
        for line in fh:
            if not line.strip():
                continue
            vals = line.split()
            row = dict(zip(header, vals))
            rr = int(float(row["rr"]))
            params[rr] = {k: float(v) for k, v in row.items()}
    return params


fit_params = load_fit_params(FIT_PARAMS_FILE)


def fit_legend_lines(cname):
    """Estrae il numero canvas da cname (es. 'c_fit_12' -> 12) e ritorna le
    righe mpv / sigma Landau / sigma gauss (prima convoluzione landau-gauss)
    da inserire in legenda. Ritorna [] se non disponibili."""
    m = re.search(r"(\d+)$", cname)
    if not m:
        return []
    rr = int(m.group(1))
    row = fit_params.get(rr)
    if row is None:
        return []
    return [
        f"mpv = {row['landau_mpv']:.4g} ± {row['e_landau_mpv']:.2g}",
        f"sigma Landau = {row['landau_width']:.4g} ± {row['e_landau_width']:.2g}",
        f"sigma gauss = {row['gauss_width']:.4g} ± {row['e_gauss_width']:.2g}",
    ]


def graph_to_arrays(g):
    n = g.GetN()
    x = np.frombuffer(g.GetX(), dtype=np.float64, count=n).copy()
    y = np.frombuffer(g.GetY(), dtype=np.float64, count=n).copy()
    return x, y


def hist_to_arrays(h):
    nbins = h.GetNbinsX()
    x = np.array([h.GetBinCenter(i) for i in range(1, nbins + 1)])
    y = np.array([h.GetBinContent(i) for i in range(1, nbins + 1)])
    yerr = np.array([h.GetBinError(i) for i in range(1, nbins + 1)])
    return x, y, yerr


def tf1_to_arrays(func, x_min=None, x_max=None, npoints=500):
    if x_min is None:
        x_min = func.GetXmin()
    if x_max is None:
        x_max = func.GetXmax()
    x = np.linspace(x_min, x_max, npoints)
    y = np.array([func.Eval(xi) for xi in x])
    return x, y


for cname in canvas_names:
    c = f.Get(cname)
    prims = c.GetListOfPrimitives()

    graphs, hists, funcs = [], [], []
    hist_funcs = {}  # nome histo -> TF1 agganciata (per calcolo residui)
    for obj in prims:
        cls = obj.ClassName()
        if cls.startswith("TGraph"):
            graphs.append(obj)
        elif cls.startswith("TH1") or cls.startswith("TH2"):
            hists.append(obj)
            # TH1 può portare la TF1 di fit agganciata nella sua lista di funzioni
            flist = obj.GetListOfFunctions()
            if flist:
                for fo in flist:
                    if fo.ClassName().startswith("TF1"):
                        funcs.append(fo)
                        hist_funcs.setdefault(obj.GetName(), fo)
        elif cls.startswith("TF1"):
            funcs.append(obj)

    print(f"\n{cname}: {len(graphs)} TGraph, {len(hists)} TH1, {len(funcs)} TF1")

    # residui = dati istogramma - fit, calcolabili solo se c'e' almeno un
    # istogramma e almeno una TF1 (uso la funzione agganciata all'istogramma
    # se esiste, altrimenti la prima TF1 disponibile)
    has_residuals = len(hists) > 0 and len(funcs) > 0

    if has_residuals:
        fig, (ax, ax_res) = plt.subplots(
            2, 1, figsize=(7, 6), sharex=True,
            gridspec_kw={"height_ratios": [0.75, 0.25], "hspace": 0})
    else:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax_res = None

    # istogramma
    for h in hists:
        x, y, yerr = hist_to_arrays(h)
        ax.errorbar(x, y, yerr=yerr, fmt="o", ms=3, label=h.GetName())
        #np.savetxt(os.path.join(OUT_DIR, f"{cname}_hist_{h.GetName()}.csv"),
        #           np.column_stack([x, y, yerr]), delimiter=",",
        #           header="x,y,yerr", comments="")

        func_for_resid = hist_funcs.get(h.GetName(), funcs[0] if funcs else None)
        if ax_res is not None and func_for_resid is not None:
            y_fit_at_bins = np.array([func_for_resid.Eval(xi) for xi in x])
            resid = y - y_fit_at_bins
            ax_res.errorbar(x, resid, yerr=yerr, fmt="o", ms=3,
                             label=f"{h.GetName()} - {func_for_resid.GetName()}")
            #np.savetxt(os.path.join(OUT_DIR, f"{cname}_resid_{h.GetName()}.csv"),
            #           np.column_stack([x, resid, yerr]), delimiter=",",
            #           header="x,residual,yerr", comments="")

    graps_labels = ['total fit', r'stopping Landau $\otimes$ Gauss', r'interacting Landau $\otimes$ Gauss' ]
    # graph (es. dati vs sistematiche, o punti di fit)
    for gg,g in enumerate(graphs):
        x, y = graph_to_arrays(g)
        ax.plot(x, y, "-", label=graps_labels[gg])

        ax.tick_params(axis="x", labelsize=14)
        ax.tick_params(axis="y", labelsize=14)
        ax.set_ylabel('counts (area normalized)',fontsize=18)

        #np.savetxt(os.path.join(OUT_DIR, f"{cname}_graph_{g.GetName()}.csv"),
        #           np.column_stack([x, y]), delimiter=",",
        #           header="x,y", comments="")

    # funzione di fit
    #for func in funcs:
    #    x, y = tf1_to_arrays(func)
    #    ax.plot(x, y, "--", label=f"fit: {func.GetName()}")
    #    np.savetxt(os.path.join(OUT_DIR, f"{cname}_fit_{func.GetName()}.csv"),
    #               np.column_stack([x, y]), delimiter=",",
    #               header="x,y", comments="")
    #    # parametri del fit
    #    npar = func.GetNpar()
    #    with open(os.path.join(OUT_DIR, f"{cname}_fit_{func.GetName()}_params.txt"), "w") as pf:
    #        for i in range(npar):
    #            pf.write(f"{func.GetParName(i)} = {func.GetParameter(i)} +- {func.GetParError(i)}\n")

    if ax_res is not None:
        # niente label ne' tick sull'asse x della pad superiore, attaccata a quella dei residui
        ax.tick_params(axis="x", labelbottom=False, bottom=False, top=False)
        ax.set_xlabel('dE/dx [MeV/cm]', fontsize=18)
        ax_res.set_xlabel('dE/dx [MeV/cm]', fontsize=18)
        ax_res.tick_params(axis="x", labelsize=14)
        ax_res.tick_params(axis="y", labelsize=14)
        ax_res.axhline(0, color="gray", lw=0.8, ls=":")
        ax_res.set_ylabel("residuals",fontsize=18)
        #ax_res.legend(fontsize=7, loc="upper left")
        ax_res.set_xlabel(ax.get_xlabel())

    # parametri del fit (mpv, sigma Landau, sigma gauss) da fitDATA.txt,
    # aggiunti come voci "invisibili" cosi' compaiono dentro la stessa legenda
    for line in fit_legend_lines(cname):
        ax.plot([], [], " ", label=line)

    rr_bin = cname.split('_')[-1]
    ax.set_title(rf'$\mathbf{{PROTONS}}$ $\mathbf{{MC\,OVERLAYS}}$ - RR bin [{int(rr_bin)},{int(rr_bin)+1})', fontsize=18)
    ax.legend(loc='upper right', fontsize=8)
    fig.subplots_adjust(hspace=0)
    fig.savefig(os.path.join(OUT_DIR, f"{cname}.pdf"), bbox_inches="tight")
    plt.close(fig)

print(f"\nFatto. Plot e CSV salvati in ./{OUT_DIR}/")

Trovati 24 canvas: ['c_fit_1', 'c_fit_2', 'c_fit_3', 'c_fit_4', 'c_fit_5', 'c_fit_6', 'c_fit_7', 'c_fit_8', 'c_fit_9', 'c_fit_10', 'c_fit_11', 'c_fit_12', 'c_fit_13', 'c_fit_14', 'c_fit_15', 'c_fit_16', 'c_fit_17', 'c_fit_18', 'c_fit_19', 'c_fit_20', 'c_fit_21', 'c_fit_22', 'c_fit_23', 'c_fit_24']

c_fit_1: 3 TGraph, 1 TH1, 1 TF1

c_fit_2: 3 TGraph, 1 TH1, 1 TF1

c_fit_3: 3 TGraph, 1 TH1, 1 TF1

c_fit_4: 3 TGraph, 1 TH1, 1 TF1

c_fit_5: 3 TGraph, 1 TH1, 1 TF1

c_fit_6: 3 TGraph, 1 TH1, 1 TF1

c_fit_7: 3 TGraph, 1 TH1, 1 TF1

c_fit_8: 3 TGraph, 1 TH1, 1 TF1

c_fit_9: 3 TGraph, 1 TH1, 1 TF1

c_fit_10: 3 TGraph, 1 TH1, 1 TF1

c_fit_11: 3 TGraph, 1 TH1, 1 TF1

c_fit_12: 3 TGraph, 1 TH1, 1 TF1

c_fit_13: 3 TGraph, 1 TH1, 1 TF1

c_fit_14: 3 TGraph, 1 TH1, 1 TF1

c_fit_15: 3 TGraph, 1 TH1, 1 TF1

c_fit_16: 3 TGraph, 1 TH1, 1 TF1

c_fit_17: 3 TGraph, 1 TH1, 1 TF1

c_fit_18: 3 TGraph, 1 TH1, 1 TF1

c_fit_19: 3 TGraph, 1 TH1, 1 TF1

c_fit_20: 3 TGraph, 1 TH1, 1 TF1

c_fit_21: 3 TGraph, 1 TH1, 1 

In [5]:
import ROOT

f = ROOT.TFile.Open("ROOT_TREES_DEDX/OUT_VARIATIONS_STANDARD_RUN2_MU_DATA.root")
c = f.Get("fit_1")

def dump(obj, indent=0):
    print("  " * indent + f"{obj.GetName()!r}  class={obj.ClassName()}")
    # se ha una lista di primitives (TPad/TCanvas), scendi dentro
    if hasattr(obj, "GetListOfPrimitives"):
        for sub in obj.GetListOfPrimitives():
            dump(sub, indent + 1)

dump(c)

'fit_1'  class=TCanvas
  'fit_1_1'  class=TPad
    'frame_x_600007d88180'  class=TH1D
    'data'  class=RooHist
    'fit'  class=RooCurve
    'langau_paramBox'  class=TPaveText
    'frame_x_600007d88180'  class=TH1D
  'fit_1_2'  class=TPad
    'resid_data_fit'  class=RooHist


In [43]:
"""
Estrae da OUT_VARIATIONS_STANDARD_RUN2_MU_DATA.root i contenuti di ciascun
canvas RooFit (fit_1 ... fit_24): RooCurve (pdf/fit), RooHist (dati), ed
eventuali istogrammi standalone (mediana_*).

Richiede PyROOT (RooFit incluso) installato.

Uso:
    python3 extract_roofit_plots.py
"""

import ROOT
import numpy as np
import matplotlib.pyplot as plt
import os

INPUT_FILE = "ROOT_TREES_DEDX/OUT_VARIATIONS_STANDARD_RUN2_MU_DATA.root"
OUT_DIR = "extracted_roofit_plots_DATA"

#INPUT_FILE = "ROOT_TREES_DEDX/OUT_VARIATIONS_STANDARD_RUN2_MU_MC.root"
#OUT_DIR = "extracted_roofit_plots_MC"
os.makedirs(OUT_DIR, exist_ok=True)

f = ROOT.TFile.Open(INPUT_FILE)
if not f or f.IsZombie():
    raise RuntimeError(f"Impossibile aprire {INPUT_FILE}")

canvas_names = [k.GetName() for k in f.GetListOfKeys() if k.GetClassName() == "TCanvas"]
hist_names = [k.GetName() for k in f.GetListOfKeys() if k.GetClassName().startswith("TH1")]
print(f"Trovati {len(canvas_names)} canvas: {canvas_names}")
print(f"Trovati {len(hist_names)} istogrammi standalone: {hist_names}")


def roocurve_to_arrays(curve):
    n = curve.GetN()
    x = np.frombuffer(curve.GetX(), dtype=np.float64, count=n).copy()
    y = np.frombuffer(curve.GetY(), dtype=np.float64, count=n).copy()
    return x, y


def roohist_to_arrays(rh):
    # RooHist e' un TGraphAsymmErrors
    n = rh.GetN()
    x = np.frombuffer(rh.GetX(), dtype=np.float64, count=n).copy()
    y = np.frombuffer(rh.GetY(), dtype=np.float64, count=n).copy()
    exl = np.frombuffer(rh.GetEXlow(), dtype=np.float64, count=n).copy()
    exh = np.frombuffer(rh.GetEXhigh(), dtype=np.float64, count=n).copy()
    eyl = np.frombuffer(rh.GetEYlow(), dtype=np.float64, count=n).copy()
    eyh = np.frombuffer(rh.GetEYhigh(), dtype=np.float64, count=n).copy()
    return x, y, exl, exh, eyl, eyh


def filter_fit_legend_lines(param_text):
    """Tiene solo le righe del TPaveText RooFit (es. langau_paramBox)
    relative a mpv, sigmaL e sigmaG, le rinomina in 'sigma Landau' /
    'sigma gauss' e converte il simbolo TLatex '#pm' in '±'."""
    rename = {"mpv": "mpv", "sigmal": "sigma Landau", "sigmag": "sigma gauss"}
    out = []
    for line in param_text:
        if "=" not in line:
            continue
        name, val = line.split("=", 1)
        key = name.strip().lower()
        if key not in rename:
            continue
        val = val.replace("#pm", "±").strip()
        val = " ".join(val.split())  # normalizza spazi multipli
        out.append(f"{rename[key]} = {val}")
    return out


def get_pads(c):
    """Ritorna la lista dei TPad figli del canvas (fit_1_1, fit_1_2, ...).
    Se il canvas non e' diviso in sotto-pad, ritorna [c] stesso."""
    pads = [obj for obj in c.GetListOfPrimitives() if obj.ClassName() == "TPad"]
    return pads if pads else [c]


def extract_pad(pad, cname, ax, label_suffix=""):
    """Scorre i primitives diretti del pad (RooCurve, RooHist, TPaveText)
    e li disegna/salva. Il TH1D 'frame_*' e' solo la cornice degli assi,
    lo si usa per i titoli ma non lo si plotta come dato."""
    n_curve, n_hist = 0, 0
    frame_hist = None
    param_text = []

    for obj in pad.GetListOfPrimitives():
        cls = obj.ClassName()
        name = obj.GetName()

        if cls == "RooCurve":
            x, y = roocurve_to_arrays(obj)
            ax.plot(x, y, "-", label=f"{name}{label_suffix}")
            #np.savetxt(os.path.join(OUT_DIR, f"{cname}_curve_{name}_{n_curve}.csv"),
            #           np.column_stack([x, y]), delimiter=",",
            #           header="x,y", comments="")
            n_curve += 1

        elif cls == "RooHist":
            x, y, exl, exh, eyl, eyh = roohist_to_arrays(obj)
            ax.errorbar(x, y, yerr=[eyl, eyh], xerr=[exl, exh],
                        fmt="o", ms=3, label=f"{name or 'data'}{label_suffix}")
            #np.savetxt(os.path.join(OUT_DIR, f"{cname}_hist_{name}_{n_hist}.csv"),
            #           np.column_stack([x, y, exl, exh, eyl, eyh]), delimiter=",",
            #           header="x,y,exlow,exhigh,eylow,eyhigh", comments="")
            n_hist += 1

        elif cls.startswith("TH1") and frame_hist is None:
            # e' la cornice/assi, la usiamo solo per i titoli
            frame_hist = obj

        elif cls == "TPaveText":
            for line in obj.GetListOfLines():
                param_text.append(line.GetTitle())
            # salva anche su file i parametri del fit (utile per langau_paramBox ecc.)
            #with open(os.path.join(OUT_DIR, f"{cname}_{name}.txt"), "w") as pf:
            #    pf.write("\n".join(param_text))

    return n_curve, n_hist, frame_hist, param_text


def render_canvas(cname, axes, show_title=True):
    """Disegna il canvas RooFit `cname` sugli assi matplotlib forniti."""

    c = f.Get(cname)
    pads = get_pads(c)

    assert len(pads) == len(axes), \
        f"{cname}: attesi {len(pads)} assi, ricevuti {len(axes)}"

    total_curve, total_hist = 0, 0
    frame_hist_main = None

    for i, (pad, ax) in enumerate(zip(pads, axes)):

        ax.tick_params(axis="both", labelsize=14)

        n_curve, n_hist, frame_hist, param_text = extract_pad(
            pad,
            f"{cname}_pad{i+1}",
            ax
        )

        print(f"{cname} pad{i}: param_text grezzo =", param_text)
        print(
            f"{cname} pad{i}: righe filtrate =",
            filter_fit_legend_lines(param_text)
        )

        total_curve += n_curve
        total_hist += n_hist

        if i == 0:
            frame_hist_main = frame_hist

        if param_text:
            for line in filter_fit_legend_lines(param_text):
                ax.plot([], [], " ", label=line)

        # ========================================================
        # LABEL ASSE Y
        # ========================================================

        if i == 0:
            ax.set_ylabel("Counts", fontsize=16)
        else:
            ax.set_ylabel("Residuals", fontsize=16)

        # ========================================================
        # PAD SUPERIORE
        # ========================================================

        if i < len(axes) - 1:
            ax.tick_params(
                axis="x",
                labelbottom=False,
                bottom=False,
                top=False
            )
            ax.set_xlabel("")

        # ========================================================
        # PAD INFERIORE
        # ========================================================

        else:
            ax.set_xlabel("dE/dx [MeV/cm]", fontsize=18)

        if i == 0:
            ax.legend(fontsize=8, loc="upper right")


    # ============================================================
    # RANGE X COMUNE A TUTTI I PAD
    # ============================================================

    if frame_hist_main is not None:

        xmin = frame_hist_main.GetXaxis().GetXmin()
        xmax = frame_hist_main.GetXaxis().GetXmax()

        #for ax in axes:
        #    ax.set_xlim(xmin, xmax)

        ax.set_xlim(0,15)

    # ============================================================
    # TITOLO
    # ============================================================

    if show_title:
        ax_title = axes[0]

        # esempio: fit_1 -> 1
        fit_number = int(cname.split("_")[-1])

        ax_title.set_title(
            fr"$\mathbf{{MUONS}}$ $\mathbf{{DATA}}$ - RR bin [{fit_number,fit_number+1})",
            fontsize=18
        )

    return total_curve, total_hist


for cname in canvas_names:
    n_pads = len(get_pads(f.Get(cname)))

    if n_pads > 1:
        height_ratios = [0.75, 0.25] #+ [0.3] * (n_pads - 2)
        fig, axes = plt.subplots(n_pads, 1, figsize=(7, 6),
                                  squeeze=False, sharex=True,
                                  gridspec_kw={"height_ratios": height_ratios, "hspace": 0})
    else:
        fig, axes = plt.subplots(n_pads, 1, figsize=(7, 5), squeeze=False)
    axes = list(axes[:, 0])

    total_curve, total_hist = render_canvas(cname, axes)

    fig.subplots_adjust(hspace=0)
    fig.savefig(os.path.join(OUT_DIR, f"{cname}.pdf"), bbox_inches="tight")
    plt.close(fig)

    print(f"{cname}: {n_pads} pad, {total_curve} RooCurve, {total_hist} RooHist estratti")

# istogrammi standalone (fuori dai canvas) - questi si leggono anche solo con uproot
for hname in hist_names:
    h = f.Get(hname)
    nbins = h.GetNbinsX()
    x = np.array([h.GetBinCenter(i) for i in range(1, nbins + 1)])
    y = np.array([h.GetBinContent(i) for i in range(1, nbins + 1)])
    yerr = np.array([h.GetBinError(i) for i in range(1, nbins + 1)])

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(x, y, yerr=yerr, fmt="o", ms=3)
    ax.set_title(hname)

    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"{hname}.pdf"))
    plt.close(fig)

    np.savetxt(os.path.join(OUT_DIR, f"{hname}.csv"),
               np.column_stack([x, y, yerr]), delimiter=",",
               header="x,y,yerr", comments="")

print(f"\nFatto. Plot e CSV salvati in ./{OUT_DIR}/")



Trovati 24 canvas: ['fit_1', 'fit_2', 'fit_3', 'fit_4', 'fit_5', 'fit_6', 'fit_7', 'fit_8', 'fit_9', 'fit_10', 'fit_11', 'fit_12', 'fit_13', 'fit_14', 'fit_15', 'fit_16', 'fit_17', 'fit_18', 'fit_19', 'fit_20', 'fit_21', 'fit_22', 'fit_23', 'fit_24']
Trovati 4 istogrammi standalone: ['mediana_data', 'mediana_mc', 'mediana_data_pro', 'mediana_mc_pro']
fit_1 pad0: param_text grezzo = ['mpv =  5.546 #pm 0.061', 'sigmaG =  0.56 #pm 0.11', 'sigmaL =  0.755 #pm 0.020']
fit_1 pad0: righe filtrate = ['mpv = 5.546 ± 0.061', 'sigma gauss = 0.56 ± 0.11', 'sigma Landau = 0.755 ± 0.020']
fit_1 pad1: param_text grezzo = []
fit_1 pad1: righe filtrate = []
fit_1: 2 pad, 1 RooCurve, 2 RooHist estratti
fit_2 pad0: param_text grezzo = ['mpv =  4.379 #pm 0.016', 'sigmaG =  0.719 #pm 0.022', 'sigmaL =  0.354 #pm 0.012']
fit_2 pad0: righe filtrate = ['mpv = 4.379 ± 0.016', 'sigma gauss = 0.719 ± 0.022', 'sigma Landau = 0.354 ± 0.012']
fit_2 pad1: param_text grezzo = []
fit_2 pad1: righe filtrate = []
fit_2:

In [17]:
import ROOT

f = ROOT.TFile.Open("ROOT_TREES_DEDX/OUT_VARIATIONS_STANDARD_RUN2_MU_DATA.root")
c = f.Get("fit_1")
pad = c.GetPrimitive("fit_1_1")
box = pad.GetPrimitive("langau_paramBox")
for line in box.GetListOfLines():
    print(repr(line.GetTitle()))

'mpv =  5.546 #pm 0.061'
'sigmaG =  0.56 #pm 0.11'
'sigmaL =  0.755 #pm 0.020'
